In [ ]:

import numpy as np
import pandas as pd
np.random.seed(42)

FILES = {
    'CO':  '../data/ekstraksi_fitur_co.csv',
    'NO2': '../data/ekstraksi_fitur_no2.csv',
    'SO2': '../data/ekstraksi_fitur_so2.csv',
}

def preprocess_np(df):
    feat_cols = [c for c in df.columns if c not in ['id', 'nama', 'daerah']]
    X = df[feat_cols].copy()
    X = X.fillna(X.median(numeric_only=True))
    # Drop constant columns
    X = X.loc[:, X.nunique() > 1]
    # Z-score standardize
    mu = X.mean()
    sd = X.std().replace(0, 1)
    Xs = ((X - mu) / sd).values
    return Xs

def pca_np(Xs, n_comp):
    # Center (already done), compute SVD
    U, S, Vt = np.linalg.svd(Xs, full_matrices=False)
    return U[:, :n_comp] * S[:n_comp]

def kmeans_np(X, k=2, max_iter=300, n_init=10, seed=42):
    rng = np.random.RandomState(seed)
    best_inertia, best_labels = np.inf, None
    for _ in range(n_init):
        idx = rng.choice(len(X), k, replace=False)
        centroids = X[idx].copy()
        for _ in range(max_iter):
            dists = np.array([[np.sum((x - c)**2) for c in centroids] for x in X])
            labels = np.argmin(dists, axis=1)
            new_centroids = np.array([X[labels == j].mean(axis=0) if (labels == j).any() else centroids[j] for j in range(k)])
            if np.allclose(centroids, new_centroids):
                break
            centroids = new_centroids
        inertia = sum(np.sum((X[labels == j] - centroids[j])**2) for j in range(k))
        if inertia < best_inertia:
            best_inertia, best_labels = inertia, labels.copy()
    return best_labels

for pol, path in FILES.items():
    df = pd.read_csv(path)
    Xs = preprocess_np(df)
    n_comp = min(37, Xs.shape[0], Xs.shape[1])
    Xp = pca_np(Xs, n_comp)
    labels = kmeans_np(Xp, k=2)
    df['cluster'] = labels

    my_row = df[df['daerah'].str.contains('Sreseh|Sampang|Bundah', case=False, na=False)]
    my_cluster = int(my_row['cluster'].values[0])
    my_pos = df.index.get_loc(my_row.index[0])
    df['dist'] = [np.linalg.norm(Xp[i] - Xp[my_pos]) for i in range(len(df))]

    print(f'\n{"="*65}')
    print(f'POLUTAN {pol}')
    print(f'Kamu: {my_row["nama"].values[0]} | {my_row["daerah"].values[0]}')
    print(f'Masuk: Cluster {my_cluster}')
    print(f'{"="*65}')

    same = df[df['cluster'] == my_cluster].sort_values('dist')
    print(f'\nCluster {my_cluster} — anggota (diurutkan paling mirip ke kamu):')
    for _, r in same.iterrows():
        tag = ' ← KAMU' if ('Sreseh' in str(r['daerah']) or 'Sampang' in str(r['daerah'])) else ''
        print(f"  id={r['id']:>2}  {r['nama']:<38} | {r['daerah']:<28} dist={r['dist']:.3f}{tag}")

    other_cluster = 1 - my_cluster
    other = df[df['cluster'] == other_cluster].sort_values('dist')
    print(f'\nCluster {other_cluster} — kelompok BERBEDA:')
    for _, r in other.iterrows():
        print(f"  id={r['id']:>2}  {r['nama']:<38} | {r['daerah']}")
